# DC Optimal Power Flow (DCOPF) Example
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

In [ ]:
from pyomo.environ import *

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────
BaseMW = 100  # MVA base

# Buses
buses = [1, 2, 3]
slack_bus = 3  # reference bus (theta fixed to 0)

# Generators: {bus: (cost $/MWh, Pgmin MW, Pgmax MW)}
gen_data = {
    1: {'c': 10, 'Pgmin': 20, 'Pgmax': 70},
    3: {'c': 20, 'Pgmin': 40, 'Pgmax': 90},
}

# Loads: {bus: MW}
load_data = {2: 100}

# Branches: {k: (from_bus, to_bus, reactance x (pu), rate MW)}
branch_data = {
    1: {'from': 1, 'to': 2, 'x': 0.1, 'rate': 60},
    2: {'from': 1, 'to': 3, 'x': 0.1, 'rate': 100},
    3: {'from': 2, 'to': 3, 'x': 0.2, 'rate': 100},
}

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.N = Set(initialize=buses,             doc='Buses')
model.G = Set(initialize=gen_data.keys(),   doc='Generators (by bus)')
model.K = Set(initialize=branch_data.keys(),doc='Branches')

# Parameters
model.c     = Param(model.G, initialize={g: gen_data[g]['c']     for g in gen_data})
model.Pgmin = Param(model.G, initialize={g: gen_data[g]['Pgmin'] for g in gen_data})
model.Pgmax = Param(model.G, initialize={g: gen_data[g]['Pgmax'] for g in gen_data})

model.Pd    = Param(model.N, initialize={n: load_data.get(n, 0)  for n in buses},
                    doc='Active load at each bus (MW)')

model.x_br  = Param(model.K, initialize={k: branch_data[k]['x']    for k in branch_data})
model.rate  = Param(model.K, initialize={k: branch_data[k]['rate']  for k in branch_data})
model.fr    = Param(model.K, initialize={k: branch_data[k]['from']  for k in branch_data},
                    within=model.N)
model.to    = Param(model.K, initialize={k: branch_data[k]['to']    for k in branch_data},
                    within=model.N)

# Decision Variables
model.Pg    = Var(model.G, doc='Generator output (MW)')
model.pk    = Var(model.K, doc='Branch flow (MW)')
model.theta = Var(model.N, doc='Voltage angle (rad)')

# ── Objective: minimise total generation cost ────────────────────
model.obj = Objective(
    expr=sum(model.c[g] * model.Pg[g] for g in model.G),
    sense=minimize
)

# ── Constraints ──────────────────────────────────────────────────

# Generator capacity limits
model.genMin = Constraint(model.G,
    rule=lambda m, g: m.Pgmin[g] <= m.Pg[g])
model.genMax = Constraint(model.G,
    rule=lambda m, g: m.Pg[g] <= m.Pgmax[g])

# DC power flow equations: pk = (theta_from - theta_to) / x  * BaseMW
model.lineFlow = Constraint(model.K,
    rule=lambda m, k: m.pk[k] / BaseMW == (m.theta[m.fr[k]] - m.theta[m.to[k]]) / m.x_br[k])

# Branch thermal limits
model.branchMin = Constraint(model.K,
    rule=lambda m, k: -m.rate[k] <= m.pk[k])
model.branchMax = Constraint(model.K,
    rule=lambda m, k: m.pk[k] <= m.rate[k])

# Nodal power balance: sum(Pg at bus n) + sum(pk flowing in) - sum(pk flowing out) = Pd
def power_balance_rule(m, n):
    gen_inj    = sum(m.Pg[g]  for g in m.G if g == n)
    flow_in    = sum(m.pk[k]  for k in m.K if m.to[k]  == n)
    flow_out   = sum(m.pk[k]  for k in m.K if m.fr[k]  == n)
    return gen_inj + flow_in - flow_out == m.Pd[n]
model.powerBalance = Constraint(model.N, rule=power_balance_rule)

# Slack bus: fix reference angle to 0
model.theta[slack_bus].fix(0)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

results = solver.solve(model, tee=True)

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\n=== Generator Dispatch ===')
for g in model.G:
    print(f'  G{g} = {model.Pg[g].value:.4f} MW')

print('\n=== Branch Flows ===')
for k in model.K:
    print(f'  pk{k} (Bus {branch_data[k]["from"]}→{branch_data[k]["to"]}) = {model.pk[k].value:.4f} MW')

print('\n=== Voltage Angles ===')
for n in model.N:
    print(f'  theta{n} = {model.theta[n].value:.6f} rad')

print(f'\n=== Total Cost = ${sum(gen_data[g]["c"] * model.Pg[g].value for g in model.G):.2f} ===')